In [ ]:
import pandas as pd
import json
import collections
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import requests
from tqdm import tqdm
import anndata as ad
import numpy as np
import seaborn as sns
from pathlib import Path

In [ ]:
METADATA_DIR = "../../../data/immunotherapy_cohorts/IMVigor210/EGAD00001003977-metadata"

# run_sample.csv has semicolon-separated key=value attributes
run_sample = pd.read_csv(f"{METADATA_DIR}/run_sample.csv")

def parse_sample_attributes(attr_str):
    attrs = {}
    if pd.isna(attr_str):
        return attrs
    for item in str(attr_str).split(";"):
        if "=" in item:
            k, v = item.split("=", 1)
            attrs[k.strip()] = v.strip()
    return attrs

attrs_df = run_sample["sample_attributes"].apply(parse_sample_attributes)
attrs_df = pd.DataFrame(list(attrs_df))

# Combine with run/sample accession IDs
meta = pd.concat([run_sample[["sample_alias", "sample_accession_id", "run_accession_id"]], attrs_df], axis=1)

# All 348 samples are RNA-seq (single library strategy: RNA-Seq, TruSeq RNA Exome)
rna_meta = meta.copy()

# Cast numeric fields
for col in ["os", "censOS", "FMOne mutation burden per MB", "Neoantigen burden per MB", "sizeFactor"]:
    rna_meta[col] = pd.to_numeric(rna_meta[col], errors="coerce")

# Canonical response label: responder = CR or PR
rna_meta["response_binary"] = rna_meta["Best Confirmed Overall Response"].map(
    {"CR": 1, "PR": 1, "SD": 0, "PD": 0}  # NE -> NaN automatically
)

print(f"Total samples in metadata:         {len(meta)}")
print(f"RNA-seq samples:                   {len(rna_meta)}")
print(f"  with OS + censOS:                {rna_meta['os'].notna().sum()}")
print(f"  with Best Confirmed Response:    {rna_meta['Best Confirmed Overall Response'].notna().sum()}")
print(f"  with binary label (excl. NE):    {rna_meta['response_binary'].notna().sum()}")
print()
print("Response breakdown:")
print(rna_meta["Best Confirmed Overall Response"].value_counts(dropna=False).to_string())
print()
print("binaryResponse breakdown:")
print(rna_meta["binaryResponse"].value_counts(dropna=False).to_string())

rna_meta.head(3)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# --- Best Response distribution ---
ax = axes[0]
response_order = ["CR", "PR", "SD", "PD", "NE"]
colors = {"CR": "#2ca02c", "PR": "#98df8a", "SD": "#ffbb78", "PD": "#d62728", "NE": "#aec7e8"}
resp_counts = rna_meta["Best Confirmed Overall Response"].value_counts().reindex(response_order, fill_value=0)
bars = ax.bar(resp_counts.index, resp_counts.values,
              color=[colors[r] for r in resp_counts.index])
ax.set_title("Best Confirmed Overall Response (IMVigor210)")
ax.set_xlabel("Response")
ax.set_ylabel("Number of RNA-seq samples")
for bar, val in zip(bars, resp_counts.values):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 1,
            str(val), ha="center", va="bottom", fontsize=10)
total_evaluable = rna_meta["response_binary"].notna().sum()
orr = rna_meta["response_binary"].sum() / total_evaluable * 100
ax.set_title(f"Best Confirmed Overall Response\nORR={orr:.1f}% (n={total_evaluable} evaluable)")

# --- IC Level breakdown ---
ax = axes[1]
ic_order = ["IC0", "IC1", "IC2+"]
ic_colors = {"IC0": "#aec7e8", "IC1": "#1f77b4", "IC2+": "#003f8a"}
ic_resp = (
    rna_meta[rna_meta["Best Confirmed Overall Response"].notna()]
    .groupby(["IC Level", "Best Confirmed Overall Response"])
    .size()
    .unstack(fill_value=0)
    .reindex(index=ic_order, columns=response_order, fill_value=0)
)
ic_resp.plot(kind="bar", stacked=True, ax=ax,
             color=[colors[c] for c in ic_resp.columns], legend=True, rot=0)
ax.set_title("Best Response by IC Level")
ax.set_xlabel("IC Level")
ax.set_ylabel("Number of samples")
ax.legend(title="Response", bbox_to_anchor=(1.01, 1), loc="upper left")
for c in ax.containers:
    ax.bar_label(c, label_type="center", fmt="%d", fontsize=8,
                 labels=[v if v > 0 else "" for v in c.datavalues])

# --- OS distribution by response ---
ax = axes[2]
resp_color_map = {"CR/PR": "#2ca02c", "SD/PD": "#d62728"}
for label, color in resp_color_map.items():
    subset = rna_meta[(rna_meta["binaryResponse"] == label) & rna_meta["os"].notna()]["os"]
    ax.hist(subset, bins=20, alpha=0.6, label=f"{label} (n={len(subset)})", color=color)
ax.set_title("Overall Survival Distribution by Response")
ax.set_xlabel("OS (months)")
ax.set_ylabel("Count")
ax.legend()

plt.tight_layout()
plt.savefig("IMVigor210_cohort_summary.png", dpi=150, bbox_inches="tight")
plt.show()

# Tabular summary by IC Level
summary = rna_meta.groupby("IC Level").agg(
    n=("sample_accession_id", "count"),
    CR=("Best Confirmed Overall Response", lambda x: (x == "CR").sum()),
    PR=("Best Confirmed Overall Response", lambda x: (x == "PR").sum()),
    SD=("Best Confirmed Overall Response", lambda x: (x == "SD").sum()),
    PD=("Best Confirmed Overall Response", lambda x: (x == "PD").sum()),
    NE=("Best Confirmed Overall Response", lambda x: (x == "NE").sum()),
    missing_response=("Best Confirmed Overall Response", lambda x: x.isna().sum()),
    has_OS=("os", lambda x: x.notna().sum()),
    median_OS=("os", "median"),
    os_event_rate=("censOS", lambda x: (x == 0).sum() / x.notna().sum()),
).reindex(ic_order)
summary["ORR (%)"] = (summary["CR"] + summary["PR"]) / (summary["n"] - summary["NE"] - summary["missing_response"]) * 100
print(summary.to_string())

In [ ]:
# Join RNA-seq sample accession IDs with file accession IDs (EGAFs)
sample_file = pd.read_csv(f"{METADATA_DIR}/sample_file.csv")

rna_accessions = set(rna_meta["sample_accession_id"])
rna_files = sample_file[sample_file["sample_accession_id"].isin(rna_accessions)].copy()

print(f"RNA-seq samples:  {len(rna_accessions)}")
print(f"Fastq files (2 per sample, paired-end): {len(rna_files)}")
print(f"Unique EGAF IDs:  {rna_files['file_accession_id'].nunique()}")
print()

# Save EGAF IDs for download
# Command:
# cat /data/rbg/users/ashyamal/RNA-FM-benchmarking/data/immunotherapy_cohorts/IMVigor210/RNA/rna_seq_egaf_ids.txt | \
#   xargs -P 8 -I {} pyega3 -cf /data/rbg/users/ashyamal/credentials.json -c 2 fetch \
#     --output-dir /data/rbg/users/ashyamal/RNA-FM-benchmarking/data/immunotherapy_cohorts/IMVigor210/RNA \
#     {}
OUT_DIR = "../../../data/immunotherapy_cohorts/IMVigor210/RNA"
egaf_path = f"{OUT_DIR}/rna_seq_egaf_ids.txt"
rna_files["file_accession_id"].to_csv(egaf_path, index=False, header=False)
print(f"Saved {len(rna_files)} EGAF IDs to {egaf_path}")

# Save a full manifest with sample alias, clinical info for easy cross-referencing
# Note: sample_alias already comes from rna_files; exclude it from rna_meta to avoid duplicate columns
manifest = rna_files.merge(
    rna_meta[[
        "sample_accession_id",
        "Best Confirmed Overall Response", "response_binary",
        "binaryResponse", "os", "censOS",
        "IC Level", "TC Level", "Immune phenotype",
        "Lund2", "TCGA Subtype",
        "FMOne mutation burden per MB", "Neoantigen burden per MB",
        "Baseline ECOG Score", "Received platinum", "Met Disease Status", "Tissue"
    ]],
    on="sample_accession_id", how="left"
).sort_values(["sample_alias", "file_name"])

manifest_path = f"{OUT_DIR}/rna_seq_download_manifest.csv"
manifest.to_csv(manifest_path, index=False)
print(f"Saved download manifest to {manifest_path}")
manifest.head(6)

In [ ]:
import subprocess

egaf_path = f"{OUT_DIR}/rna_seq_egaf_ids.txt"
dataset_files_path = f"{METADATA_DIR}/dataset_files.txt"

result = subprocess.run(
    f"grep -Ff {egaf_path} {dataset_files_path} | awk '{{sum += $6}} END {{printf \"%.2f\", sum / 1e9}}'",
    shell=True, capture_output=True, text=True
)
print(f"Total RNA-seq download size: {result.stdout} GB")

In [ ]:
# Number of samples with both RNA-seq and responder information that's note NE
labeled = rna_meta[rna_meta["response_binary"].notna()]
print(f"Samples with RNA-seq + responder/non-responder label (excl. NE): {len(labeled)}")
print(f"  Responders   (CR/PR): {(labeled['response_binary'] == 1).sum()}")
print(f"  Non-responders (SD/PD): {(labeled['response_binary'] == 0).sum()}")

In [ ]:
import os
import shutil
from pathlib import Path

RNA_DIR = Path("../../../data/immunotherapy_cohorts/IMVigor210/RNA")
manifest = pd.read_csv(RNA_DIR / "rna_seq_download_manifest.csv")

skipped, moved, already_done = [], [], []

for sample_id, grp in manifest.groupby("sample_alias"):
    sample_dir = RNA_DIR / "paired" / sample_id  # <-- saves into paired/
    sample_dir.mkdir(parents=True, exist_ok=True)

    for _, row in grp.iterrows():
        egaf_dir = RNA_DIR / row["file_accession_id"]

        fastq_files = [f for f in egaf_dir.glob("*.fastq.gz") if not f.name.endswith(".md5")]
        if not fastq_files:
            skipped.append(str(egaf_dir))
            continue

        src = fastq_files[0]
        read_num = "R1" if src.name.endswith("_1.fastq.gz") else "R2"
        dst = sample_dir / f"{sample_id}_{read_num}.fastq.gz"

        if dst.exists():
            already_done.append(str(dst))
            continue

        shutil.move(str(src), str(dst))
        moved.append(str(dst))

print(f"Moved:        {len(moved)} files")
print(f"Already done: {len(already_done)} files")
print(f"Not found:    {len(skipped)} EGAF dirs (not downloaded yet)")
if skipped:
    print("  First few missing:", skipped[:5])

### Analyze Processed RNA

In [ ]:
RNA_DIR = Path("../../../data/immunotherapy_cohorts/IMVigor210/RNA")
adata = ad.read_h5ad(RNA_DIR / "IMVigor210_bulk_RNA_processed_log1p.h5ad")
print(adata)

In [ ]:
# ── Basic counts ──────────────────────────────────────────────────────────────
print(f"Samples: {adata.n_obs}   Genes: {adata.n_vars}")
print(f"\nResponse label:\n{adata.obs['response_label'].value_counts()}")
print(f"\nIC Level:\n{adata.obs['IC Level'].value_counts()}")
print(f"\nImmune phenotype:\n{adata.obs['Immune phenotype'].value_counts()}")
print(f"\nTissue:\n{adata.obs['Tissue'].value_counts()}")
print(f"\nMissing response: {adata.obs['response_label'].isna().sum()}")


In [ ]:
# ── Response distributions ────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

adata.obs["response_label"].value_counts().plot.bar(ax=axes[0], color=["#DD8452","#4C72B0"], rot=0)
axes[0].set_title("Response (R/NR)")
axes[0].set_ylabel("Count")

ct = pd.crosstab(adata.obs["IC Level"], adata.obs["response_label"])
ct.plot.bar(stacked=True, ax=axes[1], color=["#DD8452","#4C72B0"], rot=0)
axes[1].set_title("Response by IC Level")
axes[1].set_ylabel("Count")

adata.obs["os"].hist(bins=30, ax=axes[2], color="#4C72B0", edgecolor="none")
axes[2].set_title("Overall survival distribution")
axes[2].set_xlabel("OS")

plt.suptitle("IMVigor210 cohort (BLCA, Atezolizumab)", fontsize=12)
plt.tight_layout()
plt.show()


In [ ]:
# ── Expression QC ─────────────────────────────────────────────────────────────
X = adata.X

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].hist(adata.var["log_variance"], bins=80, color="#4C72B0", edgecolor="none")
axes[0].set_title("Per-gene log-variance")
axes[0].set_xlabel("log-variance")

sample_totals = X.sum(axis=1)
axes[1].hist(sample_totals, bins=40, color="#55A868", edgecolor="none")
axes[1].set_title("Per-sample total expression (log1p)")
axes[1].set_xlabel("Sum of log1p TPM")

zero_frac = (X == 0).mean(axis=1)
axes[2].hist(zero_frac, bins=40, color="#DD8452", edgecolor="none")
axes[2].set_title("Per-sample zero-gene fraction")
axes[2].set_xlabel("Fraction of genes = 0")

plt.suptitle("IMVigor210: expression QC", fontsize=12)
plt.tight_layout()
plt.show()

print(f"Mean per-sample total expression: {sample_totals.mean():.1f}")
print(f"Mean zero-gene fraction:          {zero_frac.mean():.3f}")
print(f"Genes with log_variance > 1:      {(adata.var['log_variance'] > 1).sum()}")


In [ ]:
# ── Overlap with other ICI cohorts ────────────────────────────────────────────
other_cohorts = {}
for name in ["Hugo", "Riaz", "Rose"]:
    p = Path(f"../../../data/immunotherapy_cohorts/{name}/RNA/{name}_bulk_RNA_processed_log1p.h5ad")
    if p.exists():
        other_cohorts[name] = set(ad.read_h5ad(p).var_names)

imv_genes = set(adata.var_names)
print(f"IMVigor210 genes: {len(imv_genes)}")
for name, genes in other_cohorts.items():
    shared = len(imv_genes & genes)
    print(f"  Shared with {name}: {shared} / {min(len(imv_genes), len(genes))}")


In [ ]:
adata_raw = ad.read_h5ad(RNA_DIR / "IMVigor210_bulk_RNA_processed.h5ad")
raw_tpm_sums = adata_raw.X.sum(axis=1)
print(f"Raw TPM sum per sample — mean: {raw_tpm_sums.mean():.0f}  min: {raw_tpm_sums.min():.0f}  max: {raw_tpm_sums.max():.0f}")
# Expect ~600k-900k (less than 1M because we filtered to protein-coding genes only)

In [ ]:
low_thresh = 400000  # flag samples below this
low_samples = adata_raw.obs_names[raw_tpm_sums < low_thresh]
print(f"Samples with raw TPM sum < 400k: {len(low_samples)}")
for sid in low_samples:
    resp = adata_raw.obs.loc[sid, 'response_label']
    tpm_sum = raw_tpm_sums[adata_raw.obs_names.get_loc(sid)]
    print(f"  {sid}  TPM sum={tpm_sum:.0f}  response={resp}")
